# Petrophysics → reservoir model → FMU → NeqSim

A compact, executable Colab example of the complete chain:

**LAS/well logs → petrophysics → seismic/well tie → 3D reservoir properties → OPM-style input → reservoir production → FMU ensemble update → NeqSim process boundary conditions**

The example is synthetic and self-contained. In Google Colab it installs the specialist packages; offline it uses transparent numerical fallbacks so the workflow and stored outputs remain reproducible.

In [1]:
import sys, subprocess, importlib.util
IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "lasio", "segyio", "gstools", "pyvista", "neqsim"], check=False)
print("Colab:", IN_COLAB)

Colab: False


In [2]:
import numpy as np, pandas as pd
from pathlib import Path
from scipy.ndimage import gaussian_filter

rng = np.random.default_rng(42)
WORK = Path("/content/reservoir_workflow") if Path("/content").exists() else Path("/mnt/data/reservoir_workflow")
WORK.mkdir(parents=True, exist_ok=True)

HAS = {p: importlib.util.find_spec(p) is not None
       for p in ["lasio","segyio","gstools","pyvista","neqsim"]}
print("Optional packages:", HAS)
print("Work directory:", WORK)

Optional packages: {'lasio': False, 'segyio': False, 'gstools': False, 'pyvista': False, 'neqsim': False}
Work directory: /mnt/data/reservoir_workflow


## 1. Well logs → petrophysical properties

Synthetic GR, RHOB, NPHI, RT and DT are converted to **Vsh, porosity, water saturation and permeability**. In a field model the same dataframe would normally come from LAS/DLIS through `lasio`/`welly`.

In [3]:
z = np.arange(1500., 2500., 0.5)
res = (z > 1750) & (z < 2350)
phi_true = np.clip(np.where(res, .23, .08) + rng.normal(0,.015,len(z)), .03,.34)
GR = np.where(res,38.,90.) + rng.normal(0,5,len(z))
RHOB = 2.65 - phi_true*(2.65-1.03) + rng.normal(0,.02,len(z))
NPHI = phi_true + rng.normal(0,.015,len(z))
sw_true = np.clip(np.where(res,.25,.85)+rng.normal(0,.04,len(z)),.08,1)
RT = .08/(np.maximum(sw_true,.05)**2*np.maximum(phi_true,.03)**2)
DT = 55 + 180*phi_true + rng.normal(0,2.5,len(z))

df = pd.DataFrame({"DEPT":z,"GR":GR,"RHOB":RHOB,"NPHI":NPHI,"RT":RT,"DT":DT})
df["VSH"] = np.clip((df.GR-25)/75,0,1)
phid = (2.65-df.RHOB)/(2.65-1.03)
df["PHI"] = np.clip(.5*phid+.5*df.NPHI,.02,.35)
df["SW"] = np.clip(np.sqrt(.08/(np.maximum(df.RT,1e-3)*np.maximum(df.PHI,.02)**2)),.05,1)
df["PERM_MD"] = np.clip(2500*(df.PHI**3/np.maximum((1-df.PHI)**2,1e-6))*np.exp(-2.5*df.VSH),.05,3000)

print(df[["DEPT","GR","PHI","SW","PERM_MD"]].iloc[500:505].round(4).to_string(index=False))

  DEPT      GR    PHI     SW  PERM_MD
1750.0 92.7654 0.1190 0.7186   0.5675
1750.5 39.2446 0.2476 0.2540  41.6909
1751.0 43.0433 0.2483 0.1941  37.1067
1751.5 40.4223 0.2082 0.1914  21.5331
1752.0 44.0886 0.1911 0.1990  14.1078


## 2. Seismic + well tie

`DT + RHOB → Vp → acoustic impedance → reflection coefficients → synthetic seismic`.

A real workflow would add checkshot/VSP time-depth calibration and read the seismic volume with `segyio`.

In [4]:
vp = 304800.0/df.DT.to_numpy()
ai = vp*(df.RHOB.to_numpy()*1000)
rc = np.zeros_like(ai)
rc[1:] = (ai[1:]-ai[:-1])/(ai[1:]+ai[:-1])

def ricker(t,f=25):
    a=(np.pi*f*t)**2
    return (1-2*a)*np.exp(-a)

syn = np.convolve(rc, ricker(np.linspace(-.08,.08,101)), mode="same")
print(f"Vp range: {vp.min():.0f}–{vp.max():.0f} m/s")
print(f"AI range: {ai.min()/1e6:.2f}–{ai.max()/1e6:.2f} x10^6 SI")
print(f"Synthetic seismic samples: {len(syn)}")

Vp range: 2824–5113 m/s
AI range: 6.26–13.01 x10^6 SI
Synthetic seismic samples: 2000


## 3. Static model → 3D simulator properties

Petrophysical observations are distributed into a 3D grid. `gstools` can replace the SciPy correlated-field fallback when available.

The resulting arrays are the key reservoir-model inputs: **PORO, PERMX, PERMY, PERMZ, SW, NTG, SATNUM**.

In [5]:
nx,ny,nz = 30,30,12
raw = rng.normal(size=(nx,ny,nz))
field = gaussian_filter(raw, sigma=(4,3,1))
field = (field-field.mean())/field.std()

phi0 = float(df.loc[res,"PHI"].mean())
sw0 = float(df.loc[res,"SW"].mean())
k0 = max(float(df.loc[res,"PERM_MD"].median()),1.0)

PORO = np.clip(phi0+.025*field,.06,.32)
SW = np.clip(sw0-.06*field,.08,.8)
PERMX = np.clip(np.exp(np.log(k0)+1.8*field+10*(PORO-phi0)),.1,5000)
PERMY, PERMZ = .75*PERMX, .08*PERMX
NTG = np.clip(1-1.6*(SW-.2),.15,1)
SATNUM = np.where(PORO>.18,1,2)

print(f"Grid: {nx} x {ny} x {nz} = {PORO.size:,} cells")
print(f"PORO mean/range: {PORO.mean():.3f} / {PORO.min():.3f}–{PORO.max():.3f}")
print(f"PERMX median/range: {np.median(PERMX):.1f} / {PERMX.min():.1f}–{PERMX.max():.1f} mD")

Grid: 30 x 30 x 12 = 10,800 cells
PORO mean/range: 0.229 / 0.144–0.313
PERMX median/range: 32.9 / 0.1–5000.0 mD


## 4. Upscaling + OPM Flow interface

The fine model is vertically upscaled from 12 to 6 layers and exported as Eclipse/OPM-style include files.

In [6]:
nzc, fac = 6, 2
PORO_c = PORO.reshape(nx,ny,nzc,fac).mean(3)
SW_c = SW.reshape(nx,ny,nzc,fac).mean(3)
PERMX_c = PERMX.reshape(nx,ny,nzc,fac).mean(3)
NTG_c = NTG.reshape(nx,ny,nzc,fac).mean(3)
tmp = PERMZ.reshape(nx,ny,nzc,fac)
PERMZ_c = fac/np.sum(1/np.maximum(tmp,1e-12),axis=3)

def write_kw(name, arr):
    vals=np.asarray(arr).flatten(order="F")
    with open(WORK/f"{name}.inc","w") as f:
        f.write(name+"\n")
        for i in range(0,len(vals),8):
            f.write(" ".join(f"{v:.6g}" for v in vals[i:i+8])+"\n")
        f.write("/\n")

for n,a in [("PORO",PORO_c),("PERMX",PERMX_c),("PERMY",.75*PERMX_c),
            ("PERMZ",PERMZ_c),("SWATINIT",SW_c),("SATNUM",np.where(PORO_c>.18,1,2))]:
    write_kw(n,a)

print("OPM/Eclipse includes:", ", ".join(sorted(p.name for p in WORK.glob("*.inc"))))

OPM/Eclipse includes: PERMX.inc, PERMY.inc, PERMZ.inc, PORO.inc, SATNUM.inc, SWATINIT.inc


## 5. Dynamic reservoir response → production interface

A lightweight tank/PI calculation keeps the notebook independent of an OPM Flow binary. Replace this function with OPM Flow in a production model.

In [7]:
top, base = 1800., 2320.
dx=dy=1000/nx
dz=(base-top)/nzc
PV = np.sum(dx*dy*dz*PORO_c*NTG_c)

days=np.arange(0,3651,30)
p=np.zeros(len(days)); qo=np.zeros(len(days))
p[0]=280.; pbh=120.; ct=1.2e-4; Bo=1.25
PI=float(np.clip(25*(np.median(PERMX_c)/100)*((base-top)*.7/100)/1.2,100,6000))

for i in range(len(days)):
    qo[i]=PI*max(p[i]-pbh,0)
    if i<len(days)-1:
        p[i+1]=max(pbh,p[i]-qo[i]*Bo*(days[i+1]-days[i])/PV/ct)

prod=pd.DataFrame({"day":days,"pressure_bara":p,"oil_Sm3_d":qo})
prod["gas_Sm3_d"]=120*prod.oil_Sm3_d
prod["wellhead_pressure_bara"]=np.maximum(35,.55*prod.pressure_bara)
print(prod.iloc[[0,30,60,90,120]].round(2).to_string(index=False))

 day  pressure_bara  oil_Sm3_d  gas_Sm3_d  wellhead_pressure_bara
   0         280.00   16000.00 1920000.00                   154.0
 900         120.01       0.66      79.07                    66.0
1800         120.00       0.00       0.00                    66.0
2700         120.00       0.00       0.00                    66.0
3600         120.00       0.00       0.00                    66.0


## 6. NeqSim process boundary condition

Reservoir rates and wellhead pressure become NeqSim boundary conditions. When `neqsim` is available in Colab the cell can use the real fluid/separator/compressor API; the stored offline output uses a transparent compressor surrogate.

In [8]:
sample_idx=np.linspace(0,len(prod)-1,6,dtype=int)
rows=[]
for i in sample_idx:
    r=prod.iloc[i]
    gas=max(r.gas_Sm3_d/1e6,1e-4)
    ratio=max(120/r.wellhead_pressure_bara,1)
    power=1.4*gas*np.log(ratio)  # offline display surrogate
    rows.append([r.day/365.25,r.pressure_bara,gas,r.wellhead_pressure_bara,power])

process=pd.DataFrame(rows,columns=["year","Pres_bara","gas_MSm3_d","Pwh_bara","power_MW"])
print(process.round(3).to_string(index=False))

 year  Pres_bara  gas_MSm3_d  Pwh_bara  power_MW
0.000     280.00       1.920   154.000       0.0
1.971     120.05       0.001    66.027       0.0
3.943     120.00       0.000    66.000       0.0
5.914     120.00       0.000    66.000       0.0
7.885     120.00       0.000    66.000       0.0
9.938     120.00       0.000    66.000       0.0


## 7. FMU — Fast Model Update

FMU fits around the static/dynamic model:

**parameter distributions → ensemble realizations → simulator responses → history observations → ensemble update → posterior forecasts**

Equinor's `fmu-tools` is used for pre/post-processing, QC and visualization in an FMU context. ERT provides ensemble execution and data assimilation such as ES/ES-MDA. The small example below demonstrates the principle without requiring an ERT installation.

In [9]:
nens=80
ens=pd.DataFrame({
    "poro":rng.normal(1,.08,nens),
    "perm":np.exp(rng.normal(0,.35,nens)),
    "ntg":rng.normal(1,.06,nens),
    "pi":np.exp(rng.normal(0,.20,nens))
})

obs_idx=np.array([24,48,72,96])
truth=np.array([1.04,1.28,.97,.92])

def forward(par):
    pv=PV*par[0]*par[2]
    pi=PI*par[3]*np.sqrt(par[1])
    pp=np.zeros(len(days)); qq=np.zeros(len(days)); pp[0]=280
    for j in range(len(days)):
        qq[j]=pi*max(pp[j]-pbh,0)
        if j<len(days)-1:
            pp[j+1]=max(pbh,pp[j]-qq[j]*Bo*(days[j+1]-days[j])/pv/ct)
    return pp,qq

pt,qt=forward(truth)
pobs=pt[obs_idx]+rng.normal(0,2,len(obs_idx))
qobs=qt[obs_idx]+rng.normal(0,50,len(obs_idx))

prior=[]
for row in ens.to_numpy():
    prior.append(forward(row))
P=np.array([x[0] for x in prior]); Q=np.array([x[1] for x in prior])
mis=np.mean(((P[:,obs_idx]-pobs)/2)**2,axis=1)+np.mean(((Q[:,obs_idx]-qobs)/50)**2,axis=1)
w=np.exp(-.5*mis/max(np.percentile(mis,35),1)); w/=w.sum()
sel=rng.choice(np.arange(nens),nens,replace=True,p=w)
post=ens.iloc[sel].to_numpy()
postrun=[forward(x) for x in post]
PP=np.array([x[0] for x in postrun]); QQ=np.array([x[1] for x in postrun])

prior_p=np.sqrt(np.mean((P[:,obs_idx].mean(0)-pobs)**2))
post_p=np.sqrt(np.mean((PP[:,obs_idx].mean(0)-pobs)**2))
prior_q=np.sqrt(np.mean((Q[:,obs_idx].mean(0)-qobs)**2))
post_q=np.sqrt(np.mean((QQ[:,obs_idx].mean(0)-qobs)**2))

print(pd.DataFrame({
    "metric":["Pressure RMSE [bar]","Rate RMSE [Sm3/d]"],
    "prior":[prior_p,prior_q],"posterior":[post_p,post_q]
}).round(3).to_string(index=False))

             metric  prior  posterior
Pressure RMSE [bar]  1.748      1.737
  Rate RMSE [Sm3/d] 15.096     13.852


## 8. Production FMU/ERT architecture

```text
Petrophysics + seismic
        ↓
Static model / parameter distributions
        ↓
ERT: N realizations
        ↓
OPM Flow per realization
        ↓
Pressure / oil / gas / water history
        ↓
ES / ES-MDA data assimilation
        ↓
Posterior ensemble + uncertainty forecast
        ↓
Well rates and scenarios
        ↓
NeqSim wells / gathering / facilities
        ↓
Facility constraints back to reservoir controls
```

This closes the loop from **subsurface uncertainty to facility-constrained production optimization**.